# MalariaScope — train on a free Colab GPU

Runtime → Change runtime type → **T4 GPU**. Total time ≈ 20–30 min for EfficientNet-B0.

The numbers this notebook prints are your **real** results — put those (and only those) in the README and on your resume.

In [ ]:
!git clone https://github.com/aryanxwork/malariascope.git
%cd malariascope/ml
!pip -q install -e ".[train]"

## 1. Data (27,558 images, ~337 MB)
If the NIH server is slow/unreachable, use the Kaggle cell below instead (needs a Kaggle API token).

In [ ]:
!python scripts/download_data.py --out data

In [ ]:
# Alternative: Kaggle mirror
# from google.colab import files; files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d iarunava/cell-images-for-detecting-malaria -p data --unzip

In [ ]:
# Sanity check: number of images and patient groups (expect 27,558 images and ~200 groups)
from malariascope.data import scan_dataset
s = scan_dataset('data')
print(len(s), 'images,', len({x.patient for x in s}), 'patient groups')

## 2. Train
Run the baseline first (what the original project was), then the main model, then the leakage ablation. Each run prints a results block and saves figures under `artifacts/runs/<run>/figures/`.

In [ ]:
!python -m malariascope.train --config configs/baseline_cnn.yaml --data-dir data --output-dir ../artifacts/baseline

In [ ]:
!python -m malariascope.train --config configs/efficientnet_b0.yaml --data-dir data --output-dir ../artifacts

In [ ]:
# Optional but recommended: quantifies how much a naive random split inflates accuracy
!python -m malariascope.train --config configs/ablation_cell_split.yaml --data-dir data --output-dir ../artifacts/ablation

## 3. Export to TFLite + benchmark

In [ ]:
import glob, os
run = sorted(glob.glob('../artifacts/runs/*'), key=os.path.getmtime)[-1]
!python -m malariascope.export --model-dir ../artifacts/latest --run-dir {run} --n 500

## 4. Look at the figures, then download the model

In [ ]:
from IPython.display import Image, display
for f in ['confusion_matrix', 'roc_curve', 'reliability', 'training_curves', 'gradcam_gallery']:
    p = f'../artifacts/latest/figures/{f}.png'
    if os.path.exists(p): display(Image(p))

In [ ]:
!cd .. && zip -qr malariascope_artifacts.zip artifacts/latest artifacts/runs -x '*.csv'
from google.colab import files
files.download('../malariascope_artifacts.zip')